# 4f. Margin analysis

**What an investor reads into this.** Gross margin shows how much of each £ of revenue is left to pay for sales, development and overheads, and it is what the buyer's return is built on. The mix matters: software lines at around 80% margin are worth far more than services at around 30%, so revenue growth from services dilutes the business even when the headline grows. Discounting shows how much list price is being given away and to whom, which is a lever a new owner can pull or a sign of weak pricing power.

Definitions: `docs/metric_definitions.md` (margin section). Revenue is cube net revenue; cost is `costs.csv` total cost of delivery, which ties to management cost of sales in every month (Step 3).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nb_utils import connect, q, sql_file, save_table
import charts as ch

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
con = connect()

## 1. Gross margin by product line and year, reconciled to management gross profit

In [2]:
line = q(con, """
    WITH r AS (SELECT year(month_start) AS year, product_line, sum(net_revenue) AS revenue FROM fact_revenue_monthly GROUP BY ALL),
         c AS (SELECT year(month) AS year, product_line, sum(total_cost_of_delivery) AS cost_of_delivery,
                      sum(hosting_cost) AS hosting_cost, sum(support_staff_cost) AS support_staff_cost,
                      sum(third_party_licence_cost) AS third_party_licence_cost
               FROM clean_costs GROUP BY ALL)
    SELECT year, product_line, revenue, cost_of_delivery, revenue - cost_of_delivery AS gross_profit,
           100 * (revenue - cost_of_delivery) / revenue AS gross_margin_pct,
           hosting_cost, support_staff_cost, third_party_licence_cost
    FROM r JOIN c USING (year, product_line) ORDER BY product_line, year
""")
save_table(line, "4f_margin_by_line")
line.pivot(index="product_line", columns="year", values="gross_margin_pct").round(1)

year,2022,2023,2024,2025
product_line,,,,
Analytics Add-on,77.9,77.7,77.5,77.7
Core Platform,81.6,81.7,81.8,82.1
Implementation Services,26.7,28.7,29.8,27.1
Payments Module,65.5,64.5,64.5,65.8


In [3]:
blend = line.groupby("year")[["revenue", "cost_of_delivery", "gross_profit"]].sum().reset_index()
blend["gross_margin_pct"] = 100 * blend.gross_profit / blend.revenue
mg = q(con, """SELECT year(month) AS year, sum(total_revenue) AS mgmt_revenue, sum(cost_of_sales) AS mgmt_cost_of_sales,
                      sum(gross_profit) AS mgmt_gross_profit FROM clean_management_accounts GROUP BY 1 ORDER BY 1""")
adj = q(con, "SELECT year, manual_adjustments_in_total FROM recon_reported_total_effects")
blend = blend.merge(mg, on="year").merge(adj, on="year")
blend["mgmt_gross_margin_pct"] = 100 * blend.mgmt_gross_profit / blend.mgmt_revenue
blend["gp_difference"] = blend.mgmt_gross_profit - blend.gross_profit
blend["gp_difference_less_manual_adjustments"] = blend.gp_difference - blend.manual_adjustments_in_total
assert (blend.gp_difference_less_manual_adjustments.abs() < 0.01).all()
save_table(blend, "4f_margin_reconciliation")
blend

,year,revenue,cost_of_delivery,gross_profit,gross_margin_pct,mgmt_revenue,mgmt_cost_of_sales,mgmt_gross_profit,manual_adjustments_in_total,mgmt_gross_margin_pct,gp_difference,gp_difference_less_manual_adjustments
0,2022,28232151.64,6367072.32,21865079.32,77.447442,28232151.64,6367072.32,21865079.32,0.0,77.447442,0.0,0.0
1,2023,32192869.95,7276948.58,24915921.37,77.395776,32174369.95,7276948.58,24897421.37,-18500.0,77.382778,-18500.0,0.0
2,2024,37258898.51,8276093.15,28982805.36,77.787607,37261398.51,8276093.15,28985305.36,2500.0,77.789097,2500.0,0.0
3,2025,45756419.35,12008133.71,33748285.64,73.756396,45771419.35,12008133.71,33763285.64,15000.0,73.764996,15000.0,0.0


The invoice-based gross profit differs from management gross profit only by the four unexplained manual adjustments to reported revenue (Step 3). Costs tie exactly.

## 2. Mix and rate effects on blended margin

In [4]:
rows = []
yrs = sorted(line.year.unique())
for y in yrs[1:]:
    a = line[line.year == y - 1].set_index("product_line"); b = line[line.year == y].set_index("product_line")
    wa, wb = a.revenue / a.revenue.sum(), b.revenue / b.revenue.sum()
    ma, mb = a.gross_margin_pct, b.gross_margin_pct
    mix = ((wb - wa) * ma).sum(); rate = (wb * (mb - ma)).sum()
    tot = (wb * mb).sum() - (wa * ma).sum()
    rows.append({"year": y, "blended_margin_prior_pct": (wa * ma).sum(), "blended_margin_pct": (wb * mb).sum(),
                 "change_pts": tot, "mix_effect_pts": mix, "rate_effect_pts": rate, "check": tot - mix - rate})
mixrate = save_table(pd.DataFrame(rows), "4f_mix_rate")
assert (mixrate.check.abs() < 1e-9).all()
mixrate

,year,blended_margin_prior_pct,blended_margin_pct,change_pts,mix_effect_pts,rate_effect_pts,check
0,2023,77.447442,77.395776,-0.051667,-0.081638,0.029971,-2.498002e-16
1,2024,77.395776,77.787607,0.391831,0.285652,0.106179,7.924217e-15
2,2025,77.787607,73.756396,-4.031211,-4.032451,0.001240,-7.965850e-15


In [5]:
mixd = []
for y in yrs[1:]:
    a = line[line.year == y - 1].set_index("product_line"); b = line[line.year == y].set_index("product_line")
    wa, wb = a.revenue / a.revenue.sum(), b.revenue / b.revenue.sum()
    for pl in b.index:
        mixd.append({"year": y, "product_line": pl, "revenue_share_prior_pct": 100 * wa[pl], "revenue_share_pct": 100 * wb[pl],
                     "mix_effect_pts": (wb[pl] - wa[pl]) * a.gross_margin_pct[pl],
                     "rate_effect_pts": wb[pl] * (b.gross_margin_pct[pl] - a.gross_margin_pct[pl])})
save_table(pd.DataFrame(mixd), "4f_mix_rate_by_line")
pd.DataFrame(mixd)[lambda d: d.year == max(yrs)]

,year,product_line,revenue_share_prior_pct,revenue_share_pct,mix_effect_pts,rate_effect_pts
8,2025,Analytics Add-on,11.187619,9.998353,-0.921771,0.022859
9,2025,Core Platform,74.441193,69.028208,-4.429278,0.168800
10,2025,Implementation Services,3.070984,11.542254,2.524079,-0.313992
11,2025,Payments Module,11.300204,9.431185,-1.205480,0.123573


## 3. Margin by customer size band

Basis A (main): each product line's monthly cost is allocated to customers pro rata to their net revenue on that line in the month. Limit: this assumes cost moves with revenue, so bands differ only through the mix of product lines they buy; a large, heavily discounted customer is not shown as costing more or less per £.

Basis B (sensitivity): each line's monthly cost is shared equally across the customers billed on that line in the month (cost per active customer line). This is closer to a cost-to-serve view for support-heavy lines, and shows how far the answer depends on the allocation basis.

In [6]:
clm = sql_file(con, "customer_line_month_revenue")
clm["cost_a"] = np.where(clm.line_revenue_month != 0, clm.line_cost_month * clm.net_revenue / clm.line_revenue_month, 0.0)
clm["cost_b"] = np.where((clm.net_revenue > 0) & (clm.line_customers_month > 0), clm.line_cost_month / clm.line_customers_month, 0.0)
clm["year"] = pd.to_datetime(clm.month_start).dt.year
sb = clm.groupby(["year", "company_size"]).agg(revenue=("net_revenue", "sum"), cost_basis_a=("cost_a", "sum"),
                                               cost_basis_b=("cost_b", "sum"), customers=("customer_id", "nunique")).reset_index()
sb["margin_basis_a_pct"] = 100 * (sb.revenue - sb.cost_basis_a) / sb.revenue
sb["margin_basis_b_pct"] = 100 * (sb.revenue - sb.cost_basis_b) / sb.revenue
# the allocated cost must add back to total cost (basis B leaves out cost in months where a line has no billed customer)
tot_cost = line.groupby("year").cost_of_delivery.sum()
chk = sb.groupby("year")[["cost_basis_a", "cost_basis_b"]].sum().join(tot_cost)
assert ((chk.cost_basis_a - chk.cost_of_delivery).abs() < 0.01).all(), chk
size_order = ["Micro (1-9)", "Small (10-49)", "Medium (50-249)", "Large (250+)"]
sb["size_order"] = sb.company_size.map({s: i for i, s in enumerate(size_order)})
sb = save_table(sb.sort_values(["year", "size_order"]).drop(columns="size_order"), "4f_margin_by_size")
sb[sb.year == max(yrs)]

,year,company_size,revenue,cost_basis_a,cost_basis_b,customers,margin_basis_a_pct,margin_basis_b_pct
14,2025,Micro (1-9),5678883.83,1.674557e+06,2.808105e+06,443,70.512568,50.551814
15,2025,Small (10-49),12761233.07,3.674484e+06,4.691364e+06,682,71.205887,63.237374
13,2025,Medium (50-249),12785193.46,3.350023e+06,3.258431e+06,415,73.797636,74.514024
12,2025,Large (250+),14531108.99,3.309070e+06,1.250233e+06,132,77.227685,91.396163


## 4. Discounting: list price against realised price

Each recurring invoice is grossed up to list-equivalent at the `discount_pct` of the contract line it bills: list-equivalent = amount / (1 − discount_pct). Revenue given up = list-equivalent − amount.

In [7]:
disc = sql_file(con, "recurring_invoice_discounts")
n_rec = q(con, "SELECT count(*) AS n FROM clean_invoices WHERE revenue_type = 'recurring' AND NOT is_credit_note").n.iloc[0]
assert len(disc) == n_rec, "every recurring invoice must match exactly one contract line"
disc["given_up"] = disc.list_equivalent - disc.amount
dy = disc.groupby("year").agg(recurring_billings=("amount", "sum"), list_equivalent=("list_equivalent", "sum"), revenue_given_up=("given_up", "sum")).reset_index()
dy["realised_pct_of_list"] = 100 * dy.recurring_billings / dy.list_equivalent
dy["discount_pct_of_list"] = 100 - dy.realised_pct_of_list
ds = disc.groupby(["year", "company_size"]).agg(recurring_billings=("amount", "sum"), list_equivalent=("list_equivalent", "sum"),
                                                revenue_given_up=("given_up", "sum")).reset_index()
ds["realised_pct_of_list"] = 100 * ds.recurring_billings / ds.list_equivalent
ds["discount_pct_of_list"] = 100 - ds.realised_pct_of_list
share = ds.groupby("year").revenue_given_up.transform("sum")
ds["share_of_revenue_given_up_pct"] = 100 * ds.revenue_given_up / share
ds["size_order"] = ds.company_size.map({s: i for i, s in enumerate(size_order)})
ds = save_table(ds.sort_values(["year", "size_order"]).drop(columns="size_order"), "4f_discount_by_size")
# margin at list against actual, whole business
m_list = dy.merge(blend[["year", "revenue", "cost_of_delivery"]], on="year")
m_list["revenue_at_list"] = m_list.revenue + m_list.revenue_given_up
m_list["gross_margin_actual_pct"] = 100 * (m_list.revenue - m_list.cost_of_delivery) / m_list.revenue
m_list["gross_margin_at_list_pct"] = 100 * (m_list.revenue_at_list - m_list.cost_of_delivery) / m_list.revenue_at_list
dy = save_table(m_list, "4f_discount_by_year")
display(dy); ds[ds.year == max(yrs)]

,year,recurring_billings,list_equivalent,revenue_given_up,realised_pct_of_list,discount_pct_of_list,revenue,cost_of_delivery,revenue_at_list,gross_margin_actual_pct,gross_margin_at_list_pct
0,2022,27577496.63,3.193068e+07,4.353185e+06,86.366764,13.633236,28232151.64,6367072.32,3.258534e+07,77.447442,80.460315
1,2023,31291719.84,3.591172e+07,4.619999e+06,87.135121,12.864879,32192869.95,7276948.58,3.681287e+07,77.395776,80.232596
2,2024,36436058.01,4.152775e+07,5.091691e+06,87.739064,12.260936,37258898.51,8276093.15,4.235059e+07,77.787607,80.458139
3,2025,40883933.63,4.623884e+07,5.354906e+06,88.419031,11.580969,45756419.35,12008133.71,5.111133e+07,73.756396,76.505924


,year,company_size,recurring_billings,list_equivalent,revenue_given_up,realised_pct_of_list,discount_pct_of_list,share_of_revenue_given_up_pct
14,2025,Micro (1-9),4740760.60,4.782417e+06,4.165591e+04,99.128978,0.871022,0.777902
15,2025,Small (10-49),10781514.19,1.112318e+07,3.416645e+05,96.928356,3.071644,6.380402
13,2025,Medium (50-249),11356310.18,1.232958e+07,9.732673e+05,92.106240,7.893760,18.175246
12,2025,Large (250+),14005348.66,1.800367e+07,3.998318e+06,77.791646,22.208354,74.666450


## 5. Charts

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
pv = line.pivot(index="year", columns="product_line", values="gross_margin_pct")
order = ["Core Platform", "Analytics Add-on", "Payments Module", "Implementation Services"]
colours = [ch.BLUE, ch.ORANGE, ch.AQUA, ch.YELLOW]
for pl, col in zip(order, colours):
    ax.plot(pv.index, pv[pl], color=col, marker="o", ms=5)
    ax.text(pv.index[-1] + 0.08, pv[pl].iloc[-1], f"{pl} {pv[pl].iloc[-1]:.1f}%", fontsize=7.5, va="center", color=ch.TEXT)
ax.plot(blend.year, blend.gross_margin_pct, color=ch.TEXT, lw=2, ls=(0, (4, 2)), marker="s", ms=4)
ax.text(blend.year.iloc[-1] + 0.08, blend.gross_margin_pct.iloc[-1] - 2.5, f"Blended {blend.gross_margin_pct.iloc[-1]:.1f}%", fontsize=7.5, color=ch.TEXT)
ax.set_xticks(pv.index); ax.set_xlim(pv.index.min() - 0.2, pv.index.max() + 1.4)
ax.set_ylim(0, 100); ax.yaxis.set_major_formatter(ch.pct_fmt())
ax.set_ylabel("Gross margin (%)"); ax.set_xlabel("Calendar year")
ax.set_title("Gross margin by product line")
ax = axes[1]
x = np.arange(len(mixrate)); w = 0.36
b1 = ax.bar(x - w / 2, mixrate.mix_effect_pts, w, color=ch.BLUE, label="Mix effect")
b2 = ax.bar(x + w / 2, mixrate.rate_effect_pts, w, color=ch.ORANGE, label="Rate effect")
for bars in (b1, b2):
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width() / 2, h + (0.08 if h >= 0 else -0.08), f"{h:+.2f}", ha="center", va="bottom" if h >= 0 else "top", fontsize=8)
ax.axhline(0, color=ch.TEXT_2, lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(mixrate.year.astype(str))
ax.set_ylabel("Change in blended gross margin (percentage points)"); ax.set_xlabel("Year (against prior year)")
ax.set_title("Blended margin change: mix against rate")
ax.legend(loc="lower left")
fig.tight_layout()
print(ch.save(fig, "4f_margin"))

outputs/charts/4f_margin.png


In [9]:
fig, ax = plt.subplots(figsize=(8, 4.2))
d = ds[ds.year == max(yrs)]
bars = ax.bar(d.company_size, d.discount_pct_of_list, color=ch.BLUE, width=0.6)
for b, (_, r) in zip(bars, d.iterrows()):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.2, f"{r.discount_pct_of_list:.1f}%\n£{r.revenue_given_up / 1e6:.2f}m given up",
            ha="center", va="bottom", fontsize=8)
ax.yaxis.set_major_formatter(ch.pct_fmt())
ax.set_ylim(0, d.discount_pct_of_list.max() * 1.35)
ax.set_ylabel("Discount against list (% of list-equivalent)"); ax.set_xlabel("Customer size band")
ax.set_title(f"Average discount by size band, recurring billings {max(yrs)}")
print(ch.save(fig, "4f_discount_by_size"))

outputs/charts/4f_discount_by_size.png
